# Rink Event Map Gallery

Visual-reasoning debugging gallery for shot events overlaid on the NHL rink surface.

- **Aggregate view**: full-dataset hexbin density (log-scaled). If normalization is working, density should be concentrated in the slot at x ≈ 70-89, y ≈ [-10, 10]. Visibly bimodal density around x = ±89 indicates a normalization regression.
- **Per-period facet**: aggregate hexbin split by period 1-4 to confirm left-right symmetry.
- **Random-game panel**: pick a random current-version game and plot its shot chart for domain-knowledge sanity checks. Rerun `show_random_game()` to pull a new one; pass `seed=` for reproducibility.

Rink-drawing, density, and scatter helpers live in `src/rink_viz.py`. DB helpers
`get_random_game_id` and `load_game_shots` live in `src/database.py`. See
`knowledge_base/wiki/methods/rink-event-visualization.md` for the methodology.

In [ ]:
import os
import sys
import sqlite3

import matplotlib.pyplot as plt
import seaborn as sns

# Path setup per CLAUDE.md convention
for _candidate in [os.path.join(os.getcwd(), "src"),
                   os.path.join(os.getcwd(), "..", "src")]:
    _candidate = os.path.abspath(_candidate)
    if os.path.isdir(_candidate) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

from database import (
    DATABASE_PATH,
    _XG_EVENT_SCHEMA_VERSION,
    get_random_game_id,
    load_game_shots,
)
from rink_viz import (
    draw_full_rink,
    plot_game_shot_chart,
    plot_shot_density,
)

sns.set_theme(style="whitegrid")

conn = sqlite3.connect(DATABASE_PATH)
conn.row_factory = sqlite3.Row
print(f"Connected to {DATABASE_PATH}")
print(f"Size: {os.path.getsize(DATABASE_PATH) / 1e6:.0f} MB")

## 1. Aggregate shot density (full dataset)

Hexbin with log-scaled counts. Goal locations are at x = ±89, so a single tight
concentration on the right end confirms normalization is flipping shots as intended.

In [ ]:
cursor = conn.cursor()
cursor.execute(
    """SELECT x_coord, y_coord, period, is_goal
       FROM shot_events
       WHERE x_coord IS NOT NULL AND y_coord IS NOT NULL
         AND event_schema_version = ?""",
    (_XG_EVENT_SCHEMA_VERSION,),
)
aggregate_shots = [
    {"x_coord": x, "y_coord": y, "period": p, "is_goal": g}
    for (x, y, p, g) in cursor.fetchall()
]
print(f"Loaded {len(aggregate_shots):,} current-version shots for aggregate view")

fig, ax = plt.subplots(figsize=(14, 6))
draw_full_rink(ax)
plot_shot_density(ax, aggregate_shots, method="hexbin")
ax.set_title(
    f"All shots ({len(aggregate_shots):,}) — hexbin density, log-scaled",
    fontsize=13,
)
plt.tight_layout()
plt.show()

## 2. Per-period facet

Splits the aggregate hexbin by regulation period (1-3) and OT (period 4).
Each panel should show roughly the same tight concentration on the attacking
side. A period with a visibly different footprint is a normalization red flag
for that period or for an era where the API's `homeTeamDefendingSide` field
is missing.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for period, ax in zip([1, 2, 3, 4], axes.flatten()):
    period_shots = [s for s in aggregate_shots if s["period"] == period]
    draw_full_rink(ax)
    if period_shots:
        plot_shot_density(ax, period_shots, method="hexbin", colorbar=False)
    ax.set_title(f"Period {period} — {len(period_shots):,} shots", fontsize=11)
plt.tight_layout()
plt.show()

## 3. Random game — visual-reasoning sanity check

Pulls a random game that has at least `min_shots` current-version shot events.
Rerun the cell for a new game, or pass `seed=` to `show_random_game(seed=42)`
for reproducibility. The printed header shows the game date, teams, and shot
count so you can cross-check against your hockey knowledge.

In [ ]:
def show_random_game(season=None, min_shots=20, seed=None):
    """Draw the shot chart for a random game and return its game_id."""
    game_id = get_random_game_id(
        conn, season=season, min_shots=min_shots, seed=seed,
    )
    if game_id is None:
        print("No eligible games found for the given filters.")
        return None

    shots = load_game_shots(conn, game_id)
    if not shots:
        print(f"Game {game_id} has no current-version shot rows.")
        return game_id

    meta = shots[0]
    goals = sum(1 for s in shots if s["is_goal"])
    header = (
        f"Game {game_id} — {meta['game_date']} — "
        f"home {meta['home_team_id']} vs away {meta['away_team_id']} — "
        f"{len(shots)} shots, {goals} goals"
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    plot_game_shot_chart(ax, shots, full_rink=True)
    ax.set_title(header, fontsize=11)
    plt.tight_layout()
    plt.show()
    return game_id


show_random_game(min_shots=30);

Rerun the cell above to pull a new random game, or call
`show_random_game(season="20242025", min_shots=30)` for a filtered pick, or
`show_random_game(seed=42)` for a reproducible one.